# Backtest Report

This notebook generates CSV data (if needed), runs the backtest, and shows metrics + an equity curve.


In [ ]:
import os
from data_generator import generate_market_csv

CSV_PATH = 'market_data.csv'

if not os.path.exists(CSV_PATH):
    generate_market_csv(symbol='AAPL', start_price=150.0, filename=CSV_PATH, num_ticks=500, volatility=0.02, interval=0.0)
    print('Generated', CSV_PATH)
else:
    print('Found existing', CSV_PATH)


In [ ]:
from data_loader import load_market_data
from engine import BacktestEngine
from reporting import periodic_returns, total_return, sharpe_ratio, max_drawdown
from strategies import MovingAverageCrossoverStrategy, MomentumStrategy

ticks = load_market_data(CSV_PATH)
symbol = ticks[0].symbol

strategies = [
    MovingAverageCrossoverStrategy(symbol=symbol, short_window=5, long_window=20, order_size=10),
    MomentumStrategy(symbol=symbol, lookback=10, threshold=0.01, order_size=10),
]

engine = BacktestEngine(strategies=strategies, initial_cash=10_000.0, fail_rate=0.02)
result = engine.run(ticks)

equity_curve = result.equity_curve
initial_eq = equity_curve[0][1]
final_eq = equity_curve[-1][1]
rets = periodic_returns(equity_curve)

metrics = {
    'total_return': total_return(initial_eq, final_eq),
    'sharpe': sharpe_ratio(rets),
    'max_drawdown': max_drawdown(equity_curve),
}
metrics


In [ ]:
# Equity curve plot (matplotlib is optional)
vals = [eq for _, eq in equity_curve]

try:
    import matplotlib.pyplot as plt
    plt.figure()
    plt.plot(vals)
    plt.title('Equity Curve')
    plt.xlabel('Tick')
    plt.ylabel('Equity')
    plt.show()
except Exception as e:
    print('Plot skipped:', e)


## Narrative

- **Total return** summarizes the change in equity from start to finish.
- **Sharpe ratio** here is computed from tick-to-tick returns (not annualized). Use it to compare runs on the same data/settings.
- **Max drawdown** is the worst peak-to-trough decline in the equity curve.
